### 2.0 Setup

In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent
src_path = root/'src'
sys.path.insert(0,str(src_path))
import numpy as np
import matplotlib.pyplot as plt
import wandb
from utils.data_loader import load_fashion_mnist,load_mnist
from ann.neural_network import NeuralNetwork
from sklearn.model_selection import train_test_split

In [2]:
PROJECT_NAME = "DA6401-Assignment-1"

In [3]:
class MockArgs:
    """A simple class to mimic the argspace namespace."""
    pass

def sweep_train():
    wandb.init()
    config = wandb.config

    args = MockArgs()
    args.loss = config.loss
    args.learning_rate = config.learning_rate
    args.weight_decay = config.weight_decay
    args.optimizer = config.optimizer
    args.activation = config.activation
    args.weight_init = config.weight_init
    args.hidden_size = [int(x) for x in config.layer_config.split('_')]
    args.num_layers = len([int(x) for x in config.layer_config.split('_')])
    x_train,y_train,x_test,y_test = load_mnist()
    x_train,x_val,y_train,y_val = train_test_split(x_train,y_train,test_size=0.1,random_state= 6)
    model = NeuralNetwork(args)
    model.train(x_train, y_train, x_val, y_val, config.epochs, config.batch_size)

### 2.1 Data Exploration and Class Distribution

In [ ]:
wandb.init(project= PROJECT_NAME, name= 'question_2.1')
print('loading MINST dataset')
x_train,y_train,x_test,y_test = load_mnist()
columns = ["Class Label"] + [f"Sample {i+1}" for i in range(5)]
table = wandb.Table(columns=columns)
print("collecting samples")
for label in range(10):
    samples = np.where(y_train == label)[0]
    first5samples = samples[:5]
    row_data = [str(label)]
    for sample in first5samples:
        flat_img = x_train[sample]
        img = flat_img.reshape(28,28) * 255
        row_data.append(wandb.Image(img,caption=f"{label}"))
    table.add_data(*row_data)
wandb.log({
    "sample_data": table
})
wandb.finish()
print("finished logging samples to wandb")

### 2.2 Hyperparameter sweep

In [ ]:
sweep_config = {
    'method': 'random',
    'metric': {
        'name': 'val_accuracy',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {'values': [0.1, 0.01, 0.001, 0.0001]},
        'batch_size': {'values': [32, 64, 128, 256]},
        'optimizer': {'values': ['sgd', 'momentum', 'nag', 'rmsprop', 'adam', 'nadam']},
        'activation': {'values': ['relu', 'tanh', 'sigmoid']},
        'weight_init': {'values': ['random', 'xavier']},
        'weight_decay': {'values': [0.0001, 0.001, 0.01, 0]},
        'layer_config': {'values': ['64','128', '64_64','128_64', '64_32' , '128_128_64' ,'128_64_32', '128_128_64_32', '128_128_128_128_128', '128_64_64_64_32_16']},
        'epochs': {'values': [10]},
        'loss': {'values': ['mean_squared_error', 'cross_entropy']}
    }
}

In [ ]:
sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(sweep_id, sweep_train, count=100)

### 2.3 Optimizer showdown

In [6]:
optimizers = ['sgd', 'momentum', 'nag', 'rmsprop', 'adam', 'nadam']

x_train,y_train,x_test,y_test = load_mnist()
x_train,x_val,y_train,y_val = train_test_split(x_train,y_train,test_size=0.1,random_state= 6)

for optim in optimizers:
    print(f"\n--- Testing Optimizer: {optim.upper()} ---")
    wandb.init(
        project = PROJECT_NAME,
        name = f"2.3_optimizer_{optim}",
        group = "optimizer_showdown",
        config = {
            "optimizer": optim,
            "epochs": 5,
            "num_layers": 3,
            "hidden_size": [128, 128, 128],
            "activation": "relu"
        }
    )

    args = MockArgs()
    args.dataset = 'mnist'
    args.loss = 'cross_entropy'
    args.learning_rate = 0.001
    args.weight_decay = 0.0
    args.optimizer = optim
    args.activation = 'relu'
    args.weight_init = 'random'
    args.num_layers = 3
    args.hidden_size = [128, 128, 128]

    model = NeuralNetwork(args)
    model.train(x_train, y_train, x_val, y_val,epochs = 5,batch_size= 128)

    wandb.finish()


--- Testing Optimizer: SGD ---


Epoch 1/5 - Train Loss: 7.8921 - Val Acc: 0.5053
Epoch 2/5 - Train Loss: 4.9250 - Val Acc: 0.6513
Epoch 3/5 - Train Loss: 4.0388 - Val Acc: 0.7158
Epoch 4/5 - Train Loss: 3.6566 - Val Acc: 0.7437


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 3.4205 - Val Acc: 0.7578


epoch,▁▃▅▆█
first_layer_grad_norm,█▅▃▁▁
layer:0_dead_neuron_pct,▁▄▅▆█
layer:1_dead_neuron_pct,▁▅▆▇█
layer:2_dead_neuron_pct,▁▃▅▇█
train_accuracy,▁▅▇▇█
train_loss,█▃▂▁▁
val_accuracy,▁▅▇██
val_loss,█▃▂▁▁
epoch,5
first_layer_grad_norm,4.62346


...


--- Testing Optimizer: NADAM ---


Epoch 1/5 - Train Loss: 0.6212 - Val Acc: 0.8853
Epoch 2/5 - Train Loss: 0.2737 - Val Acc: 0.9240
Epoch 3/5 - Train Loss: 0.1734 - Val Acc: 0.9345
Epoch 4/5 - Train Loss: 0.1167 - Val Acc: 0.9433


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.0771 - Val Acc: 0.9462


epoch,▁▃▅▆█
first_layer_grad_norm,█▂▄▁▁
layer:0_dead_neuron_pct,▁▆██▅
layer:1_dead_neuron_pct,▁▅▇▇█
layer:2_dead_neuron_pct,▁▇▇██
train_accuracy,▁▅▆▇█
train_loss,█▄▂▂▁
val_accuracy,▁▅▇██
val_loss,█▃▂▁▁
epoch,5
first_layer_grad_norm,0.79424


### 2.4 Vanishing gradient analysis

In [9]:
activations = ['relu', 'sigmoid']
depths = [
    (2,[128,128]),
    (3,[128,128,128]),
    (4,[128,128,128,128])
]

x_train,y_train,x_test,y_test = load_mnist()
x_train,x_val,y_train,y_val = train_test_split(x_train,y_train,test_size=0.1,random_state= 6)

for activation in activations:
    print(f"\n--- Testing Activation: {activation.upper()} ---")
    for depth,sizes in depths:
        wandb.init(
            project = PROJECT_NAME,
            name = f"2.4__{activation}_depth:{depth}",
            group = "activation_showdown",
            config = {
                "optimizer": 'adam',
                "epochs": 5,
                "num_layers": depth,
                "hidden_size": sizes,
                "activation": activation
            }
        )

        args = MockArgs()
        args.dataset = 'mnist'
        args.loss = 'cross_entropy'
        args.learning_rate = 0.001
        args.weight_decay = 0.0
        args.optimizer = 'adam'
        args.activation = activation
        args.weight_init = 'random'
        args.num_layers = depth
        args.hidden_size = sizes

        model = NeuralNetwork(args)
        model.train(x_train, y_train, x_val, y_val,epochs = 5,batch_size= 128)

    wandb.finish()


--- Testing Activation: RELU ---


Epoch 1/5 - Train Loss: 0.4114 - Val Acc: 0.9007
Epoch 2/5 - Train Loss: 0.2378 - Val Acc: 0.9257
Epoch 3/5 - Train Loss: 0.1536 - Val Acc: 0.9378
Epoch 4/5 - Train Loss: 0.1079 - Val Acc: 0.9448
Epoch 5/5 - Train Loss: 0.0817 - Val Acc: 0.9487


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▃▅▆█
first_layer_grad_norm,█▆▁▅▁
layer:0_dead_neuron_pct,▁▇▆█▇
layer:1_dead_neuron_pct,▁▇▆█▇
train_accuracy,▁▄▆▇█
train_loss,█▄▃▂▁
val_accuracy,▁▅▆▇█
val_loss,█▄▂▁▁
epoch,5
first_layer_grad_norm,0.42117
layer:0_dead_neuron_pct,60.86077


Epoch 1/5 - Train Loss: 0.7486 - Val Acc: 0.8892
Epoch 2/5 - Train Loss: 0.3579 - Val Acc: 0.9157
Epoch 3/5 - Train Loss: 0.2188 - Val Acc: 0.9278
Epoch 4/5 - Train Loss: 0.1614 - Val Acc: 0.9335
Epoch 5/5 - Train Loss: 0.1029 - Val Acc: 0.9408


...

epoch,▁▃▅▆█
first_layer_grad_norm,▅▅▆█▁
layer:0_dead_neuron_pct,▁▁▁▅█
layer:1_dead_neuron_pct,▁▁▁▁▁
layer:2_dead_neuron_pct,▁▁▁▁▁
train_accuracy,▁▄▆▇█
train_loss,█▅▃▂▁
val_accuracy,▁▅▆██
val_loss,█▄▃▂▁
epoch,5
first_layer_grad_norm,0.11835


Epoch 1/5 - Train Loss: 0.2561 - Val Acc: 0.9278
Epoch 2/5 - Train Loss: 0.1661 - Val Acc: 0.9497
Epoch 3/5 - Train Loss: 0.1321 - Val Acc: 0.9542
Epoch 4/5 - Train Loss: 0.0992 - Val Acc: 0.9623


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.0894 - Val Acc: 0.9612


epoch,▁▃▅▆█
first_layer_grad_norm,▅▁▃▁█
layer:0_dead_neuron_pct,▁▁▁▁█
layer:1_dead_neuron_pct,▁▁▁▁▁
layer:2_dead_neuron_pct,▁▁▁▁▁
layer:3_dead_neuron_pct,▁▁▁▁▁
train_accuracy,▁▅▆██
train_loss,█▄▃▁▁
val_accuracy,▁▅▆██
val_loss,█▄▂▁▁
epoch,5


### 2.5 Dead neuron investigation

In [10]:
activations = ['relu', 'tanh']
depths = [(3,[128,128,128])]

x_train,y_train,x_test,y_test = load_mnist()
x_train,x_val,y_train,y_val = train_test_split(x_train,y_train,test_size=0.1,random_state= 6)

for activation in activations:
    print(f"\n--- Testing Activation: {activation.upper()} ---")
    for depth,sizes in depths:
        wandb.init(
            project = PROJECT_NAME,
            name = f"2.5__{activation}",
            group = "dead_neuron_investigation",
            config = {
                "optimizer": 'sgd',
                "epochs": 15,
                "num_layers": depth,
                "hidden_size": sizes,
                "activation": activation
            }
        )

        args = MockArgs()
        args.dataset = 'mnist'
        args.loss = 'cross_entropy'
        args.learning_rate = 0.1
        args.weight_decay = 0.0
        args.optimizer = 'sgd'
        args.activation = activation
        args.weight_init = 'random'
        args.num_layers = depth
        args.hidden_size = sizes

        model = NeuralNetwork(args)
        model.train(x_train, y_train, x_val, y_val,epochs = 15,batch_size= 128)

    wandb.finish()


--- Testing Activation: RELU ---


Epoch 1/15 - Train Loss: 0.3139 - Val Acc: 0.9028
Epoch 2/15 - Train Loss: 0.1995 - Val Acc: 0.9275
Epoch 3/15 - Train Loss: 0.1612 - Val Acc: 0.9345
Epoch 4/15 - Train Loss: 0.1192 - Val Acc: 0.9442
Epoch 5/15 - Train Loss: 0.0979 - Val Acc: 0.9492
Epoch 6/15 - Train Loss: 0.0878 - Val Acc: 0.9497
Epoch 7/15 - Train Loss: 0.0768 - Val Acc: 0.9507
Epoch 8/15 - Train Loss: 0.0693 - Val Acc: 0.9538
Epoch 9/15 - Train Loss: 0.0599 - Val Acc: 0.9537
Epoch 10/15 - Train Loss: 0.0461 - Val Acc: 0.9563
Epoch 11/15 - Train Loss: 0.0412 - Val Acc: 0.9555
Epoch 12/15 - Train Loss: 0.0329 - Val Acc: 0.9595
Epoch 13/15 - Train Loss: 0.0330 - Val Acc: 0.9588
Epoch 14/15 - Train Loss: 0.0296 - Val Acc: 0.9573


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 15/15 - Train Loss: 0.0224 - Val Acc: 0.9587


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
first_layer_grad_norm,█▆▆▄▃▂▆▂▁▂▁▃▅▃▃
layer:0_dead_neuron_pct,█▇▆▅▄▄▃▃▃▂▂▂▁▁▁
layer:1_dead_neuron_pct,█▇▇▅▅▄▄▃▄▃▂▂▂▁▁
layer:2_dead_neuron_pct,▅██▅▅▅▅▄▃▃▄▁▂▁▁
train_accuracy,▁▃▄▅▆▆▇▇▇▇█████
train_loss,█▅▄▃▃▃▂▂▂▂▁▁▁▁▁
val_accuracy,▁▄▅▆▇▇▇▇▇██████
val_loss,█▅▃▂▂▂▁▂▁▁▁▁▁▁▁
epoch,15
first_layer_grad_norm,0.38627



--- Testing Activation: TANH ---


Epoch 1/15 - Train Loss: 0.3482 - Val Acc: 0.8798
Epoch 2/15 - Train Loss: 0.2428 - Val Acc: 0.9052
Epoch 3/15 - Train Loss: 0.1915 - Val Acc: 0.9160
Epoch 4/15 - Train Loss: 0.1549 - Val Acc: 0.9262
Epoch 5/15 - Train Loss: 0.1343 - Val Acc: 0.9308
Epoch 6/15 - Train Loss: 0.1138 - Val Acc: 0.9338
Epoch 7/15 - Train Loss: 0.0983 - Val Acc: 0.9362
Epoch 8/15 - Train Loss: 0.0859 - Val Acc: 0.9377
Epoch 9/15 - Train Loss: 0.0758 - Val Acc: 0.9380
Epoch 10/15 - Train Loss: 0.0680 - Val Acc: 0.9395
Epoch 11/15 - Train Loss: 0.0608 - Val Acc: 0.9403
Epoch 12/15 - Train Loss: 0.0538 - Val Acc: 0.9410
Epoch 13/15 - Train Loss: 0.0485 - Val Acc: 0.9432
Epoch 14/15 - Train Loss: 0.0429 - Val Acc: 0.9413


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 15/15 - Train Loss: 0.0386 - Val Acc: 0.9443


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
first_layer_grad_norm,█▇▅▇▄▃▇▃▅▁▅▃▂▃▄
layer:0_dead_neuron_pct,▁▆▆▆█████▆▆▆███
layer:1_dead_neuron_pct,▂▁▂▁▄▁▅▅██▇█▇█▇
layer:2_dead_neuron_pct,█▁▁▁▅▁▅▅▅▅▅▅▅▅▅
train_accuracy,▁▃▄▅▆▆▇▇▇▇█████
train_loss,█▆▄▄▃▃▂▂▂▂▂▁▁▁▁
val_accuracy,▁▄▅▆▇▇▇▇▇▇█████
val_loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁
epoch,15
first_layer_grad_norm,0.42412


### 2.6 Loss function comparsion

In [12]:
loss_functions = ['cross_entropy', 'mean_squared_error']

x_train,y_train,x_test,y_test = load_mnist()
x_train,x_val,y_train,y_val = train_test_split(x_train,y_train,test_size=0.1,random_state= 6)

for loss in loss_functions:
    print(f"\n--- Testing Loss function: {loss.upper()} ---")
    wandb.init(
        project = PROJECT_NAME,
        name = f"2.6_lossfunction_{loss}",
        group = "loss_function",
        config = {
            "optimizer": 'adam',
            "epochs": 5,
            "num_layers": 3,
            "hidden_size": [128, 128, 128],
            "activation": "relu",
            "loss": loss
        }
    )

    args = MockArgs()
    args.dataset = 'mnist'
    args.loss = loss
    args.learning_rate = 0.001
    args.weight_decay = 0.0
    args.optimizer = 'adam'
    args.activation = 'relu'
    args.weight_init = 'random'
    args.num_layers = 3
    args.hidden_size = [128, 128, 128]

    model = NeuralNetwork(args)
    model.train(x_train, y_train, x_val, y_val,epochs = 5,batch_size= 128)

    wandb.finish()


--- Testing Loss function: CROSS_ENTROPY ---


Epoch 1/5 - Train Loss: 0.6522 - Val Acc: 0.8873
Epoch 2/5 - Train Loss: 0.3334 - Val Acc: 0.9142
Epoch 3/5 - Train Loss: 0.2059 - Val Acc: 0.9250
Epoch 4/5 - Train Loss: 0.1481 - Val Acc: 0.9342


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.1015 - Val Acc: 0.9408


epoch,▁▃▅▆█
first_layer_grad_norm,█▅▄▁▄
layer:0_dead_neuron_pct,▁▆█▇▇
layer:1_dead_neuron_pct,▁▇▇█▇
layer:2_dead_neuron_pct,▁▆▇█▇
train_accuracy,▁▄▆▇█
train_loss,█▄▂▂▁
val_accuracy,▁▅▆▇█
val_loss,█▄▂▂▁
epoch,5
first_layer_grad_norm,1.11421



--- Testing Loss function: MEAN_SQUARED_ERROR ---


Epoch 1/5 - Train Loss: 0.0924 - Val Acc: 0.5198
Epoch 2/5 - Train Loss: 0.0694 - Val Acc: 0.6343
Epoch 3/5 - Train Loss: 0.0515 - Val Acc: 0.7257
Epoch 4/5 - Train Loss: 0.0386 - Val Acc: 0.7913


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch 5/5 - Train Loss: 0.0318 - Val Acc: 0.8203


epoch,▁▃▅▆█
first_layer_grad_norm,▆▁██▄
layer:0_dead_neuron_pct,▁▄▇██
layer:1_dead_neuron_pct,▁▄▅██
layer:2_dead_neuron_pct,▁▅▆█▇
train_accuracy,▁▄▆▇█
train_loss,█▅▃▂▁
val_accuracy,▁▄▆▇█
val_loss,█▅▃▂▁
epoch,5
first_layer_grad_norm,0.08457


### 2.7 